In [2]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10793358 sha256=7cb5dd37e4eef46a3fc4f0d8e70870fbb31a314cbd9daf3d3284a657a4c464a4
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [16]:
!pip install implicit pandarallel -q

import os, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRanker, Pool
import implicit
from tqdm.auto import tqdm

# Для текстов
from pandarallel import pandarallel
import nltk
import re
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import SnowballStemmer

pandarallel.initialize(progress_bar=True, nb_workers=os.cpu_count())

# Конфиг
DATA_DIR    = "/kaggle/input/datasets/artemnazemtsev/nto-ai"
OUTPUT_PATH = "submission.csv"

TOP_K               = 20
CANDIDATES_PER_USER = 300
INCIDENT_DAYS       = 30
HOLDOUT_DAYS        = 30
RANDOM_SEED         = 42

ALS_FACTORS = 64
ALS_ITERS   = 30

print("Библиотеки и конфиг загружены!")

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Библиотеки и конфиг загружены!


In [17]:
def optimize_dtypes(df):
    for col in df.columns:
        if df[col].dtype == 'int64':
            df[col] = df[col].astype(np.int32)
        elif df[col].dtype == 'float64':
            df[col] = df[col].astype(np.float32)
    return df

print("Загрузка данных...")
interactions = pd.read_csv(f"{DATA_DIR}/interactions.csv", parse_dates=["event_ts"])
targets      = pd.read_csv(f"{DATA_DIR}/targets.csv")
editions     = pd.read_csv(f"{DATA_DIR}/editions.csv")
book_genres  = pd.read_csv(f"{DATA_DIR}/book_genres.csv")
users        = pd.read_csv(f"{DATA_DIR}/users.csv")

# Оптимизация памяти
interactions = optimize_dtypes(interactions)
targets      = optimize_dtypes(targets)
editions     = optimize_dtypes(editions)
book_genres  = optimize_dtypes(book_genres)
users        = optimize_dtypes(users)
interactions['event_type'] = interactions['event_type'].astype(np.int8)

T_MAX            = interactions["event_ts"].max()
T_INCIDENT_START = T_MAX - pd.Timedelta(days=INCIDENT_DAYS)
T_TRAIN_SPLIT    = T_INCIDENT_START - pd.Timedelta(days=HOLDOUT_DAYS)

print(f"Interactions: {len(interactions):,}")
print(f"Сплит трейна: {T_TRAIN_SPLIT.date()}, Начало инцидента: {T_INCIDENT_START.date()}")

Загрузка данных...
Interactions: 443,278
Сплит трейна: 2025-10-01, Начало инцидента: 2025-10-31


In [18]:
RU_STOP = set(nltk_sw.words("russian"))
EN_STOP = set(nltk_sw.words("english"))
STOPWORDS = RU_STOP | EN_STOP
STEMMER = SnowballStemmer("russian")

_RE_HTML  = re.compile(r"<[^>]+>")
_RE_CHARS = re.compile(r"[^а-яёa-z\s]")

def stem_and_filter(text: str, max_tokens: int = 200) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = [STEMMER.stem(t) for t in text.split() if len(t) >= 3 and t not in STOPWORDS]
    return " ".join(tokens[:max_tokens])

def vector_clean(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str)
    s = s.str.normalize("NFKC").str.replace(_RE_HTML, " ", regex=True).str.lower().str.replace(_RE_CHARS, " ", regex=True)
    return s

print("Очистка текстов...")
t_prep = vector_clean(editions["title"])
d_prep = vector_clean(editions["description"])

print("Стемминг (многопоточно)...")
editions["title_clean"] = t_prep.parallel_apply(stem_and_filter)
editions["desc_clean"]  = d_prep.parallel_apply(lambda x: stem_and_filter(x, max_tokens=120))

ed_avg_rating_map = (
    interactions[interactions["event_type"] == 2]
    .groupby("edition_id")["rating"].mean().fillna(0).to_dict()
)

Очистка текстов...
Стемминг (многопоточно)...


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Вычисление TF-IDF для описаний...")
# Объединяем название и описание для лучшего контекста
editions['full_text'] = editions['title_clean'] + " " + editions['desc_clean']

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(editions['full_text'].fillna(""))

# Создаем быстрый маппинг: edition_id -> индекс в матрице
eid_to_tfidf_idx = {eid: i for i, eid in enumerate(editions['edition_id'])}

print("TF-IDF готов.")

Вычисление TF-IDF для описаний...
TF-IDF готов.


In [27]:
def build_features(iact: pd.DataFrame, T_split):
    pos = iact[iact["event_type"].isin([1, 2])].copy()
    
    # 1. Базовые счетчики
    ep = pos.groupby("edition_id").size().rename("edition_pop")
    ep_w = pos[pos["event_type"] == 1].groupby("edition_id").size().rename("wishlist_pop")
    ep_r = pos[pos["event_type"] == 2].groupby("event_type").size().rename("read_pop") # Fix: groupby edition_id
    ep_r = pos[pos["event_type"] == 2].groupby("edition_id").size().rename("read_pop")
    bgc = book_genres.groupby("book_id").size().rename("book_genre_count")
    
    # 2. "Горячая" популярность (последние 14 дней в данном сплите)
    recent_cutoff = T_split - pd.Timedelta(days=14)
    recent_pop = pos[pos["event_ts"] >= recent_cutoff].groupby("edition_id").size().rename("recent_pop")
    
    ed_feat = editions[["edition_id", "book_id", "author_id", "publication_year", "language_id", "title_clean", "desc_clean"]].set_index("edition_id")
    ed_feat = ed_feat.join(ep).join(ep_w).join(ep_r).join(recent_pop).join(bgc, on="book_id").fillna(0)
    ed_feat["pop_log"] = np.log1p(ed_feat["edition_pop"]).astype(np.float32)
    
    # 3. Фичи пользователей + АГРЕГАТЫ ПО ИСТОРИИ
    u_total = pos.groupby("user_id").size().rename("user_activity")
    
    # Сливаем историю с фичами изданий, чтобы понять, ЧТО читает юзер
    hist_stats = pos[["user_id", "edition_id"]].merge(
        ed_feat[["publication_year", "edition_pop", "recent_pop"]].reset_index(), 
        on="edition_id"
    )
    u_hist_agg = hist_stats.groupby("user_id").agg({
        "publication_year": ["mean", "std"],
        "edition_pop": ["mean", "max"],
        "recent_pop": "mean"
    })
    u_hist_agg.columns = ["_".join(c) for c in u_hist_agg.columns]
    
    u_feat = users[["user_id", "gender", "age"]].set_index("user_id")
    u_feat = u_feat.join(u_total).join(u_hist_agg).fillna(0)
    u_feat["user_activity_log"] = np.log1p(u_feat["user_activity"]).astype(np.float32)
    
    # 4. Профили (жанры и авторы)
    df_g = pos.merge(editions[["edition_id", "book_id"]], on="edition_id").merge(book_genres, on="book_id")
    ugp = df_g.groupby(["user_id", "genre_id"]).size().rename("cnt").reset_index()
    ugp["user_genre_affinity"] = (ugp["cnt"] / ugp.groupby("user_id")["cnt"].transform("sum")).astype(np.float32)
    
    df_a = pos.merge(editions[["edition_id", "author_id"]], on="edition_id").dropna(subset=["author_id"])
    uap = df_a.groupby(["user_id", "author_id"]).size().rename("cnt").reset_index()
    uap["user_author_affinity"] = (uap["cnt"] / uap.groupby("user_id")["cnt"].transform("sum")).astype(np.float32)
    
    return ed_feat, u_feat, ugp[["user_id", "genre_id", "user_genre_affinity"]], uap[["user_id", "author_id", "user_author_affinity"]], pos

In [34]:
from implicit.nearest_neighbours import bm25_weight

def train_als_and_get_matrix(pos_iact: pd.DataFrame):
    pos = pos_iact.copy()
    # Веса: чтение важнее вишлиста в 3 раза
    pos["w"] = pos["event_type"].map({1: 1.0, 2: 3.0})

    ue_enc, ei_enc = LabelEncoder(), LabelEncoder()
    u_idx = ue_enc.fit_transform(pos["user_id"])
    e_idx = ei_enc.fit_transform(pos["edition_id"])

    # Создаем матрицу
    user_item_csr = sparse.csr_matrix(
        (pos["w"].values, (u_idx, e_idx)), 
        shape=(len(ue_enc.classes_), len(ei_enc.classes_))
    )
    
    # ЖЕСТКИЙ МЕТОД №1: BM25 Weighting перед ALS
    # Это заставляет модель обращать внимание на редкие книги, а не только на хиты
    user_item_csr = bm25_weight(user_item_csr, K1=100, B=0.8).tocsr()

    als = implicit.als.AlternatingLeastSquares(
        factors=128, # Увеличили размерность для более тонких связей
        iterations=40, 
        regularization=0.01,
        use_gpu=True, 
        random_state=RANDOM_SEED
    )
    als.fit(user_item_csr) 

    U = als.user_factors.to_numpy().astype(np.float32)
    V = als.item_factors.to_numpy().astype(np.float32)
    
    # Центроиды для I2I (расстояние до среднего вектора интересов)
    U_hist_centroid = (user_item_csr.dot(V) / np.array(user_item_csr.getnnz(axis=1)).reshape(-1, 1).clip(min=1)).astype(np.float32)
    
    U_norm = U / np.linalg.norm(U, axis=1, keepdims=True).clip(min=1e-9)
    V_norm = V / np.linalg.norm(V, axis=1, keepdims=True).clip(min=1e-9)
    U_hist_norm = U_hist_centroid / np.linalg.norm(U_hist_centroid, axis=1, keepdims=True).clip(min=1e-9)

    u2als = {uid: i for i, uid in enumerate(ue_enc.classes_)}
    e2als = {eid: i for i, eid in enumerate(ei_enc.classes_)}
    
    return als, user_item_csr, U, V, U_norm, V_norm, U_hist_norm, u2als, e2als, ue_enc.classes_, ei_enc.classes_

In [ ]:
from implicit.nearest_neighbours import ItemItemRecommender, bm25_weight

def train_recommenders(pos_iact: pd.DataFrame):
    pos = pos_iact.copy()
    pos["w"] = pos["event_type"].map({1: 1.0, 2: 3.0}) # Чтение важнее вишлиста

    ue_enc, ei_enc = LabelEncoder(), LabelEncoder()
    u_idx = ue_enc.fit_transform(pos["user_id"])
    e_idx = ei_enc.fit_transform(pos["edition_id"])

    # Матрица взаимодействий
    ui_csr = sparse.csr_matrix((pos["w"].values, (u_idx, e_idx)), 
                               shape=(len(ue_enc.classes_), len(ei_enc.classes_)))
    
    # ЖЕСТКИЙ МЕТОД: BM25 нормализация
    ui_csr_bm25 = bm25_weight(ui_csr, K1=100, B=0.8).tocsr()

    # 1. ALS (Латентные факторы)
    als = implicit.als.AlternatingLeastSquares(factors=128, iterations=30, use_gpu=True)
    als.fit(ui_csr_bm25)
    
    # 2. Item-Item (Прямая схожесть соседей)
    ii_model = ItemItemRecommender(K=20)
    ii_model.fit(ui_csr_bm25.T)

    U = als.user_factors.to_numpy().astype(np.float32)
    V = als.item_factors.to_numpy().astype(np.float32)
    
    # 3. TF-IDF Центроиды юзеров (Средний текстовый профиль интересов)
    # Это матричное умножение: (User x Item) * (Item x TF-IDF_dim)
    # Показывает "о чем" обычно читает юзер
    user_counts = np.array(ui_csr.getnnz(axis=1)).reshape(-1, 1).clip(min=1)
    # Берем индексы книг из ei_enc.classes_, чтобы сопоставить с tfidf_matrix
    item_indices = [eid_to_idx[eid] for eid in ei_enc.classes_]
    relevant_tfidf = tfidf_matrix[item_indices]
    user_tfidf_centroids = (ui_csr.dot(relevant_tfidf) / user_counts).astype(np.float32)

    u2als = {uid: i for i, uid in enumerate(ue_enc.classes_)}
    e2als = {eid: i for i, eid in enumerate(ei_enc.classes_)}
    
    return als, ii_model, U, V, user_tfidf_centroids, u2als, e2als, ue_enc.classes_, ei_enc.classes_

In [29]:
def generate_candidates(targets_list, als_model, user_item_csr, u2als, e2als_arr, pop_items, ed_feat, ugp, uap, k=150):
    print("Генерация ALS U2I...")
    cands_list = []
    batch_size = 2000
    
    # 1. ALS Candidates
    for i in tqdm(range(0, len(targets_list), batch_size)):
        batch_users = targets_list[i:i+batch_size]
        u_indices = [u2als.get(u, -1) for u in batch_users]
        
        valid_mask = np.array(u_indices) != -1
        v_users = np.array(batch_users)[valid_mask]
        v_indices = np.array(u_indices)[valid_mask]
        
        if len(v_indices) == 0: continue
            
        ids, scores = als_model.recommend(v_indices, user_item_csr[v_indices], N=k, filter_already_liked_items=True)
        
        for idx, uid in enumerate(v_users):
            valid_items = ids[idx][ids[idx] != -1]
            eids = e2als_arr[valid_items]
            scrs = scores[idx][ids[idx] != -1]
            cands_list.append(pd.DataFrame({"user_id": uid, "edition_id": eids, "cand_score": scrs, "src_als": 1}))
            
    cands_df = pd.concat(cands_list, ignore_index=True) if cands_list else pd.DataFrame(columns=["user_id", "edition_id", "cand_score", "src_als"])
    
    # 2. Добивка популярными для холодных юзеров
    missing_users = set(targets_list) - set(cands_df['user_id'].unique())
    if missing_users:
        pop_cands = pd.DataFrame({
            'user_id': np.repeat(list(missing_users), len(pop_items)),
            'edition_id': np.tile(pop_items, len(missing_users)),
            'cand_score': 0.0, 'src_als': 0
        })
        cands_df = pd.concat([cands_df, pop_cands], ignore_index=True)

    return cands_df.drop_duplicates(subset=["user_id", "edition_id"])

In [35]:
def build_ranking_dataset(cands_df, u_feat, ed_feat, ugp, uap, U, V, U_tfidf, u2als, e2als, ii_model, ui_csr):
    df = cands_df.merge(u_feat, on='user_id', how='left')
    df = df.merge(ed_feat, on='edition_id', how='left')
    
    u_idx_raw = df["user_id"].map(u2als).fillna(-1).astype(int).values
    e_idx_raw = df["edition_id"].map(e2als).fillna(-1).astype(int).values
    valid = (u_idx_raw != -1) & (e_idx_raw != -1)
    
    # --- 1. ALS Метрики ---
    df["als_dot"] = 0.0
    if valid.any():
        df.loc[valid, "als_dot"] = (U[u_idx_raw[valid]] * V[e_idx_raw[valid]]).sum(axis=1)

    # --- 2. Item-Item Similarity (Графовая близость) ---
    # Насколько эта книга похожа на историю юзера по графу взаимодействий
    df["ii_score"] = 0.0
    if valid.any():
        # Быстрый расчет через скалярное произведение матрицы схожести
        # Извлекаем кусок матрицы схожести для кандидатов
        ii_sim = ii_model.similarity # Это матрица Item x Item
        # Для скорости используем упрощенный скор из implicit
        for i, row in tqdm(df[valid].iterrows(), total=valid.sum(), desc="II Scoring"):
            u_i, e_i = u2als[row['user_id']], e2als[row['edition_id']]
            # Скор = (строка юзера в ui_csr) * (столбец айтема в матрице схожести)
            df.at[i, "ii_score"] = ui_csr[u_i].dot(ii_sim[:, e_i].T)[0,0]

    # --- 3. TF-IDF Cosine (Семантическая близость) ---
    df["tfidf_cos"] = 0.0
    # Индексы в глобальной матрице TF-IDF
    global_e_idx = df["edition_id"].map(eid_to_idx).values
    for i, row in tqdm(df.iterrows(), total=len(df), desc="TF-IDF Scoring"):
        u_id = row['user_id']
        if u_id in u2als:
            u_vec = U_tfidf[u2als[u_id]]
            e_vec = tfidf_matrix[global_e_idx[i]]
            # Косинус между центром интересов юзера и книгой
            df.at[i, "tfidf_cos"] = e_vec.dot(u_vec.T)[0,0]

    # --- 4. Агрегаты по соседям (Cross-features) ---
    df["rel_pop"] = df["edition_pop"] / (df["user_activity"] + 1)
    df["year_diff"] = df["publication_year"] - df["publication_year_mean"]

    return df.fillna(0)

In [36]:
print("--- TEMPORAL SPLIT ---")
fake_history = interactions[interactions["event_ts"] < T_TRAIN_SPLIT].copy()
fake_holdout = interactions[(interactions["event_ts"] >= T_TRAIN_SPLIT) & (interactions["event_type"].isin([1, 2]))].copy()

train_users = list(set(fake_history["user_id"]) & set(fake_holdout["user_id"]))
pos_holdout_pairs = set(zip(fake_holdout["user_id"], fake_holdout["edition_id"]))

ed_feat_tr, u_feat_tr, ugp_tr, uap_tr, pos_tr = build_features(fake_history, T_TRAIN_SPLIT)
als_tr, csr_tr, U_tr, V_tr, Un_tr, Vn_tr, Uhist_tr, u2als_tr, e2als_tr, u_arr_tr, e_arr_tr = train_als_and_get_matrix(pos_tr)


pop_items_tr = fake_history['edition_id'].value_counts().head(50).index.tolist()

train_cands = generate_candidates(train_users, als_tr, csr_tr, u2als_tr, e_arr_tr, pop_items_tr, ed_feat_tr, ugp_tr, uap_tr)

# Разметка таргета
train_cands['target'] = train_cands.apply(lambda x: 1.0 if (x['user_id'], x['edition_id']) in pos_holdout_pairs else 0.0, axis=1)

train_df = build_ranking_dataset(train_cands, u_feat_tr, ed_feat_tr, ugp_tr, uap_tr, U_tr, V_tr, Un_tr, Vn_tr, Uhist_tr, u2als_tr, e2als_tr)

train_df = train_df.sort_values('user_id').reset_index(drop=True)
y_train = train_df.pop('target').values
groups_train = train_df['user_id'].values

--- TEMPORAL SPLIT ---


  0%|          | 0/40 [00:00<?, ?it/s]

Генерация ALS U2I...


  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import gc
gc.collect()

In [42]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
from tqdm.auto import tqdm

print("--- [CELL 1] VALIDATION & TF-IDF SETUP ---")

# 1. Разбивка по времени (последние 14 дней — в валидацию)
T_MAX = interactions['event_ts'].max()
T_VAL_START = T_MAX - pd.Timedelta(days=14)

train_iact = interactions[interactions['event_ts'] < T_VAL_START].copy()
val_iact = interactions[interactions['event_ts'] >= T_VAL_START].copy()

# Оставляем только "прочтения" (target=1) для оценки NDCG
val_true_pairs = set(val_iact[val_iact['event_type'] == 2][['user_id', 'edition_id']].itertuples(index=False, name=None))
val_users = [u for u in val_iact['user_id'].unique() if u in train_iact['user_id'].unique()]

# 2. TF-IDF для семантики
editions['full_text'] = (editions['title_clean'].fillna("") + " " + editions['desc_clean'].fillna("")).astype(str)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(editions['full_text']).astype(np.float32)

# Нормируем векторы для быстрого косинуса через dot-product
from sklearn.preprocessing import normalize
tfidf_matrix = normalize(tfidf_matrix, axis=1)

eid_to_idx = {eid: i for i, eid in enumerate(editions['edition_id'])}
print(f"Валидация: {len(val_users)} юзеров. TF-IDF матрица: {tfidf_matrix.shape}")

--- [CELL 1] VALIDATION & TF-IDF SETUP ---
Валидация: 5088 юзеров. TF-IDF матрица: (460599, 5000)


In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from implicit.nearest_neighbours import ItemItemRecommender, bm25_weight
from catboost import CatBoostRanker, Pool
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# --- 1. ИСПРАВЛЕННАЯ ФУНКЦИЯ ГЕНЕРАЦИИ ПРИЗНАКОВ ---
def get_advanced_features(df, u_feat, ed_feat, U_als, V_als, U_tfidf, u2als, e2als, ui_csr, ii_sim):
    """Матричный расчет фичей: ALS, TF-IDF центроиды и Item-Item Similarity"""
    
    # Принудительная конвертация всех матриц в CSR для поддержки индексации [indices]
    if not isinstance(U_tfidf, sparse.csr_matrix):
        U_tfidf = sparse.csr_matrix(U_tfidf)
    if not isinstance(ii_sim, sparse.csr_matrix):
        ii_sim = sparse.csr_matrix(ii_sim)
    if not isinstance(ui_csr, sparse.csr_matrix):
        ui_csr = sparse.csr_matrix(ui_csr)

    # Объединяем с базовыми признаками пользователей и книг
    df = df.merge(u_feat, on='user_id', how='left').merge(ed_feat, on='edition_id', how='left')
    
    u_idx = df["user_id"].map(u2als).fillna(-1).astype(int).values
    e_idx = df["edition_id"].map(e2als).fillna(-1).astype(int).values
    valid = (u_idx != -1) & (e_idx != -1)
    
    # 1. ALS Dot (Векторная близость факторов)
    df["als_dot"] = 0.0
    if valid.any():
        df.loc[valid, "als_dot"] = (U_als[u_idx[valid]] * V_als[e_idx[valid]]).sum(axis=1)
    
    # 2. TF-IDF Cosine (Семантическая близость интересов пользователя к книге)
    df["tfidf_cos"] = 0.0
    # eid_to_idx должен быть определен глобально при обучении TF-IDF
    g_e_idx = df["edition_id"].map(eid_to_idx).fillna(-1).astype(int).values
    v_tfidf = valid & (g_e_idx != -1)
    
    if v_tfidf.any():
        u_vecs = U_tfidf[u_idx[v_tfidf]]
        e_vecs = tfidf_matrix[g_e_idx[v_tfidf]]
        # Построчное скалярное произведение разреженных векторов
        df.loc[v_tfidf, "tfidf_cos"] = np.array(u_vecs.multiply(e_vecs).sum(axis=1)).flatten()
        
    # 3. Item-Item Score (Насколько книга похожа на то, что юзер УЖЕ читал)
    df["ii_score"] = 0.0
    if valid.any():
        # Считаем батчами по уникальным юзерам, чтобы не перегружать RAM
        u_ids_unique = df.loc[valid, 'user_id'].unique()
        for chunk in np.array_split(u_ids_unique, 40):
            mask = df['user_id'].isin(chunk)
            c_u = df.loc[mask, 'user_id'].map(u2als).values
            c_e = df.loc[mask, 'edition_id'].map(e2als).values
            
            # Извлекаем историю батча и столбцы схожести для кандидатов
            hist = ui_csr[c_u]
            sims = ii_sim[:, c_e].T
            # Эффективное умножение "строка на строку"
            res = hist.multiply(sims).sum(axis=1)
            df.loc[mask, "ii_score"] = np.array(res).flatten()

    return df.fillna(0)

# --- 2. ПРОЦЕСС ОБУЧЕНИЯ (ПОЛНЫЙ ЦИКЛ) ---

print("Шаг 1: Подготовка матриц и обучение моделей...")
ue_enc, ei_enc = LabelEncoder(), LabelEncoder()
u_idx_tr = ue_enc.fit_transform(train_iact["user_id"])
e_idx_tr = ei_enc.fit_transform(train_iact["edition_id"])

# Создаем базовую CSR матрицу взаимодействий
ui_csr = sparse.csr_matrix((np.ones(len(u_idx_tr)), (u_idx_tr, e_idx_tr)))
ui_csr_bm25 = bm25_weight(ui_csr, K1=100, B=0.8).tocsr()

# Обучаем ALS
als = implicit.als.AlternatingLeastSquares(factors=128, iterations=30, random_state=42)
als.fit(ui_csr_bm25)

# Обучаем Item-Item (KNN для книг)
ii_model = ItemItemRecommender(K=20)
ii_model.fit(ui_csr_bm25.T)

u2als = {uid: i for i, uid in enumerate(ue_enc.classes_)}
e2als = {eid: i for i, eid in enumerate(ei_enc.classes_)}

# Сборка профилей пользователей (U_tfidf) через матричное умножение
print("Шаг 2: Создание семантических профилей пользователей...")
item_indices_local = [eid_to_idx[eid] for eid in ei_enc.classes_]
U_tfidf = ui_csr.dot(tfidf_matrix[item_indices_local])
# Нормализация: делим вектор на количество прочитанных книг, чтобы получить среднее
user_counts = np.array(ui_csr.sum(axis=1)).flatten().clip(min=1)
U_tfidf = sparse.diags(1.0 / user_counts).dot(U_tfidf).tocsr() # ПРИНУДИТЕЛЬНО В CSR

# Сборка финального тренировочного датасета
print("Шаг 3: Генерация продвинутых признаков...")
train_df_advanced = get_advanced_features(
    train_cands, u_feat_tr, ed_feat_tr, 
    als.user_factors.to_numpy(), als.item_factors.to_numpy(),
    U_tfidf, u2als, e2als, ui_csr, ii_model.similarity
)

# --- 3. ОБУЧЕНИЕ АНСАМБЛЯ (YETIRANK) ---

def train_ensemble(df, n_models=3):
    print(f"Шаг 4: Обучение ансамбля из {n_models} моделей...")
    models = []
    text_cols = ["title_clean", "desc_clean"]
    drop_cols = ["user_id", "edition_id", "book_id", "author_id", "target"]
    feat_cols = [c for c in df.columns if c not in drop_cols + text_cols]
    
    # Создаем Pool для CatBoost один раз
    train_pool = Pool(
        data=df[feat_cols + text_cols], 
        label=df["target"], 
        group_id=df["user_id"], 
        text_features=text_cols
    )
    
    for i in range(n_models):
        print(f"Обучение модели #{i+1}...")
        m = CatBoostRanker(
            loss_function="YetiRankPairwise", 
            iterations=1200, 
            learning_rate=0.03, 
            depth=6, 
            task_type="GPU", 
            random_seed=42+i,
            verbose=200
        )
        m.fit(train_pool)
        models.append(m)
    return models, feat_cols

ensemble_models, final_feat_cols = train_ensemble(train_df_advanced)
print("Обучение завершено успешно!")

Шаг 1: Подготовка матриц и обучение моделей...


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/18650 [00:00<?, ?it/s]

Шаг 2: Создание семантических профилей пользователей...
Шаг 3: Генерация продвинутых признаков...


In [40]:
print("--- [STAGE 1] INFERENCE PREPARATION ---")
# 1. Генерируем фичи на полном логе перед окном инцидента
# (Используем pos_f, который возвращается из обновленной функции build_features)
ed_feat_f, u_feat_f, ugp_f, uap_f, pos_f = build_features(interactions, T_INCIDENT_START)

# 2. Переобучаем ALS на полном логе (с BM25 весами)
als_f, csr_f, U_f, V_f, Un_f, Vn_f, Uhist_f, u2als_f, e2als_f, u_arr_f, e_arr_f = train_als_and_get_matrix(pos_f)

# 3. Настраиваем воронку кандидатов
CANDIDATES_PER_USER = 300 # Расширяем воронку для CatBoost
pop_items_f = interactions['edition_id'].value_counts().head(100).index.tolist()
test_users = targets["user_id"].unique().tolist()

print(f"Генерация {CANDIDATES_PER_USER} кандидатов для {len(test_users)} пользователей...")
test_cands = generate_candidates(test_users, als_f, csr_f, u2als_f, e_arr_f, pop_items_f, ed_feat_f, ugp_f, uap_f, k=CANDIDATES_PER_USER)

print("Сборка датасета для ранжирования...")
test_df = build_ranking_dataset(
    test_cands, u_feat_f, ed_feat_f, ugp_f, uap_f, 
    U_f, V_f, Un_f, Vn_f, Uhist_f, u2als_f, e2als_f
)

# --- [STAGE 2] ENSEMBLE PREDICT ---
print(f"Инференс ансамбля из {len(ensemble_models)} моделей...")
test_pool = Pool(data=test_df[feat_cols + text_cols], text_features=text_cols)

all_preds = []
for i, m in enumerate(ensemble_models):
    print(f"Предсказание модели #{i+1}...")
    all_preds.append(m.predict(test_pool))

# Усредняем предсказания
test_df["score"] = np.mean(all_preds, axis=0)

# --- [STAGE 3] POST-PROCESSING & BLENDING ---
# ЖЕСТКИЙ МЕТОД: Добавляем небольшой вес за "горячую" популярность
# Это помогает вытащить потеряшки, которые были в тренде в момент сбоя
test_df["score"] = test_df["score"] + 0.07 * np.log1p(test_df["recent_pop"])

# Ранжируем
res = test_df[["user_id", "edition_id", "score"]].sort_values(["user_id", "score"], ascending=[True, False])
res = res.groupby("user_id").head(TOP_K).copy()

# --- [STAGE 4] FILLING GAPS & EXPORT ---
print("Формирование финального списка и проверка на пропуски...")
per_user = {u: g["edition_id"].tolist() for u, g in res.groupby("user_id")}
final_rows = []

for uid in tqdm(test_users):
    seen = per_user.get(uid, [])
    
    # Добавляем предсказанные
    for e in seen:
        final_rows.append({"user_id": uid, "edition_id": e})
    
    # Добивка популярными, если кандидатов не хватило (холодные юзеры)
    need = TOP_K - len(seen)
    if need > 0:
        extra = [e for e in pop_items_f if e not in seen][:need]
        for e in extra:
            final_rows.append({"user_id": uid, "edition_id": e})

# Создаем сабмит
submission = pd.DataFrame(final_rows)
submission["rank"] = submission.groupby("user_id").cumcount() + 1

# Финальная проверка структуры (должно быть ровно len(targets) * 20 строк)
expected_len = len(targets) * TOP_K
if len(submission) != expected_len:
    print(f"ВНИМАНИЕ: Ошибка длины! Ожидалось {expected_len}, получено {len(submission)}")

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Успех! Файл '{OUTPUT_PATH}' готов к отправке.")

--- [STAGE 1] INFERENCE PREPARATION ---


  0%|          | 0/40 [00:00<?, ?it/s]

Генерация 300 кандидатов для 3862 пользователей...
Генерация ALS U2I...


  0%|          | 0/2 [00:00<?, ?it/s]

Сборка датасета для ранжирования...
Инференс ансамбля из 3 моделей...
Предсказание модели #1...
Предсказание модели #2...
Предсказание модели #3...
Формирование финального списка и проверка на пропуски...


  0%|          | 0/3862 [00:00<?, ?it/s]

Успех! Файл 'submission.csv' готов к отправке.
